# Yug — quickstart

Forecast a time series in a few lines, with no training and no feature engineering.

Yug is a **zero-shot probabilistic** forecaster: you hand it history, it returns
a set of quantile levels over the horizon you ask for. It has never seen your series.

---

**Install**

```bash
pip install yug matplotlib
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from yug import YugPipeline

# A Hugging Face Hub id, or a path to a local checkpoint directory.
MODEL_ID = "birlaailabs/yug"

pipeline = YugPipeline.from_pretrained(MODEL_ID, device_map="cpu")
pipeline

## 1. A series to forecast

Daily data with a weekly cycle, a slow annual cycle, a mild upward trend and noise —
the shape of a lot of real demand data. We hold out the last 96 points as truth.

In [ ]:
HORIZON = 64

rng = np.random.default_rng(15)
t = np.arange(1000)

signal = (
    100.0  # level
    + 0.05 * t  # trend
    + 12.0 * np.sin(2 * np.pi * t / 7)  # weekly cycle
    + 25.0 * np.sin(2 * np.pi * t / 365)  # annual cycle
    + rng.normal(0, 3.0, len(t))  # noise
).astype(np.float32)

context, truth = signal[:-HORIZON], signal[-HORIZON:]
print(f"{len(context)} points of history, forecasting {HORIZON} ahead")

## 2. Forecast

`freq` is a real input to the model, not bookkeeping — it selects a learned embedding,
so passing the wrong one degrades the forecast.

In [ ]:
forecast = pipeline.predict(
    context,
    prediction_length=HORIZON,
    freq="D",
    num_samples=100,  # sampled trajectories behind the band
    seed=0,  # makes the call reproducible
)

forecast

In [ ]:
median = forecast.median
band = forecast.interval()  # widest trained pair, here 0.1 / 0.9

print("median :", np.round(median[:6], 2))
print("lower  :", np.round(band["lower"][:6], 2))
print("upper  :", np.round(band["upper"][:6], 2))

## 3. Plot it

In [ ]:
future_x = np.arange(len(context), len(context) + HORIZON)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(
    np.arange(len(context))[-250:],
    context[-250:],
    color="#3a3a3a",
    lw=1.1,
    label="history",
)
ax.plot(future_x, truth, color="#1a7f5a", lw=1.4, label="actual")
ax.plot(future_x, median, color="#c2410c", lw=1.6, label="forecast (median)")
ax.fill_between(
    future_x,
    band["lower"],
    band["upper"],
    color="#c2410c",
    alpha=0.18,
    label="80% interval",
)
ax.axvline(len(context), color="#999", ls="--", lw=1)
ax.set_xlabel("step")
ax.set_ylabel("value")
ax.set_title("Yug zero-shot forecast")
ax.legend(loc="upper left", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Score it

Two numbers worth looking at together:

- **MASE** — error relative to a seasonal-naive baseline. Below 1.0 beats it.
- **Coverage** — how often the truth actually falls inside the 80% band. It should be
  near 0.8. A sharp median with badly calibrated intervals is a worse model than the
  headline error suggests.

In [ ]:
season = 7

naive_error = np.mean(np.abs(context[season:] - context[:-season]))
mase = np.mean(np.abs(median - truth)) / naive_error
coverage = np.mean((truth >= band["lower"]) & (truth <= band["upper"]))

print(f"MAE       {np.mean(np.abs(median - truth)):8.3f}")
print(f"MASE      {mase:8.3f}   (< 1.0 beats seasonal naive)")
print(f"Coverage  {coverage:8.3f}   (nominal 0.80)")

## 5. Many series at once

Pass a list of arrays. They may have different lengths and different frequencies.

In [ ]:
series_a = context
series_b = context[:400] * 0.5 + 40
series_c = np.cumsum(rng.normal(0, 1, 600)).astype(np.float32) + 200

batch = pipeline.predict(
    [series_a, series_b, series_c],
    prediction_length=48,
    freq=["D", "D", "H"],
    num_samples=100,
    item_ids=["a", "b", "c"],
    seed=0,
)

print(batch.values.shape, " (series, quantiles, horizon)")
batch.to_dataframe().head()

### Covariates

Pass a 2-D array `(n_channels, T)` as a single series to condition on covariates. The
target channel is row 0 by default. Note that covariate rollout tiles the drawn block
across every channel, so covariate forecasts degrade faster with horizon than
univariate ones.

```python
multivariate = np.stack([target, covariate_1, covariate_2])
pipeline.predict([multivariate], prediction_length=48)
```

### A worked covariate example

The cell below forecasts a real dataset (hourly electricity prices) pulled from an
external source, both with and without covariates, and compares the two. It needs
extra dependencies not required by the rest of this notebook:

```bash
pip install pandas pyarrow
```

In [ ]:
import pandas as pd

# --- Load the Amazon electricity-price dataset (hourly, Germany) ---
context_df = pd.read_parquet(
    "https://autogluon.s3.amazonaws.com/datasets/timeseries/electricity_price/train.parquet"
)
test_df = pd.read_parquet(
    "https://autogluon.s3.amazonaws.com/datasets/timeseries/electricity_price/test.parquet"
)

print(f"Context: {context_df.shape}, Test: {test_df.shape}")

HORIZON = 24

target = context_df["target"].values.astype(np.float32)
cov_load = context_df["Ampirion Load Forecast"].values.astype(np.float32)
cov_renewables = context_df["PV+Wind Forecast"].values.astype(np.float32)

# --- 1. Univariate forecast (target only, no covariates) ---
print("Forecasting univariate (no covariates)...")
forecast_uni = pipeline.predict(
    target, prediction_length=HORIZON, context_length=8192, freq="H", num_samples=30, seed=0
)
print(f"Univariate forecast: {forecast_uni}")

# --- 2. Covariate informed forecast (past covariates only, no future leakage) ---
print("Forecasting with past covariates only (Load + Renewables)...")
multivariate = np.stack([target, cov_load, cov_renewables])
forecast_past = pipeline.predict(
    [multivariate], prediction_length=HORIZON, context_length=8192, freq="H", num_samples=30, seed=0
)
print(f"Past-covariates forecast: {forecast_past}")

# --- Plot: two-panel comparison ---
truth = test_df["target"].values
timestamps = test_df["timestamp"].values
hist_ts = context_df["timestamp"].values[-500:]
hist_vals = context_df["target"].values[-500:]

# Compute shared y-limits across both panels
all_vals = np.concatenate(
    [
        hist_vals,
        truth,
        forecast_uni.median,
        forecast_past.median,
        forecast_uni.interval()["upper"],
        forecast_past.interval()["upper"],
    ]
)
ymin, ymax = np.min(all_vals) - 5, np.max(all_vals) + 5

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

for ax, fc, title in zip(
    axes,
    [forecast_uni, forecast_past],
    [
        "Univariate (target only)",
        "With past covariates only (Load + Renewables)",
    ],
    strict=True,
):
    band = fc.interval()
    ax.plot(hist_ts, hist_vals, color="#3a3a3a", lw=0.8, label="history")
    ax.plot(timestamps, truth, color="#1a7f5a", lw=1.2, label="actual")
    ax.plot(timestamps, fc.median, color="#c2410c", lw=1.4, label="median")
    ax.fill_between(
        timestamps,
        band["lower"],
        band["upper"],
        color="#c2410c",
        alpha=0.15,
        label="80% band",
    )
    ax.axvline(timestamps[0], color="#999", ls="--", lw=1)
    ax.set_ylabel("EUR/MWh")
    ax.set_title(title)
    ax.set_ylim(ymin, ymax)
    ax.legend(loc="upper left", frameon=False)
    ax.spines[["top", "right"]].set_visible(False)

axes[1].set_xlabel("time")
fig.suptitle(
    "Yug — German electricity price (hourly)",
    y=1.01,
    fontsize=13,
)
plt.tight_layout()
plt.show()

# --- Metrics ---
season = 24
hist = context_df["target"].values
naive_denom = np.mean(np.abs(hist[season:] - hist[:-season]))
for name, fc in [
    ("Univariate", forecast_uni),
    ("Past cov only", forecast_past),
]:
    median_vals = fc.median
    mae = np.mean(np.abs(median_vals - truth))
    mase = mae / naive_denom
    band = fc.interval()
    cov = np.mean((truth >= band["lower"]) & (truth <= band["upper"]))
    print(f"{name:16s}  MAE={mae:.2f}  MASE={mase:.3f}  Coverage={cov:.2f}")